# TP 4 — Estimation de densité

**Cours :** *Density Estimation* (slides 89–117) — biais, variance, MSE ; estimateurs **paramétriques** (gaussien, mélanges de gaussiennes / EM) et **non paramétriques** (histogrammes, estimateurs à noyau) ; vitesses de convergence et malédiction de la dimension.

**À la fin de ce notebook, vous saurez :**
1. ajuster une **gaussienne** (MLE) et constater son **biais constant** quand les données ne sont pas gaussiennes — le cas des rendements financiers ;
2. coder l'algorithme **EM** pour un **mélange de gaussiennes** (GMM), choisir le nombre de composantes (BIC) et l'utiliser pour détecter des **régimes de marché** ;
3. coder un **histogramme** et un **estimateur à noyau** (KDE) en dimension quelconque, et choisir la fenêtre (règle de Silverman, validation croisée) ;
4. vérifier numériquement les **vitesses de convergence** du cours et la malédiction de la dimension ;
5. comparer ces estimateurs sur un problème de risque : la **Value-at-Risk** à 99 % du S&P 500, avec backtest.

> 📝 **Version énoncé.** Complétez les blocs `# TODO`, puis exécutez la cellule *Test* qui suit chaque exercice. Le corrigé est dans `../corriges/`.

In [ ]:
import sys, pathlib, warnings
_here = pathlib.Path.cwd()
ROOT = _here.parent if _here.name in ("exercices", "corriges") else _here
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tp_utils import *          # style graphique, check(), check_close(), timer()...
import tp_data                  # données réelles (cache / téléchargement) ou simulées

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
rng = np.random.default_rng(42)
from scipy import stats
from scipy.special import logsumexp, gamma
from scipy.optimize import brentq
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KernelDensity

## Partie 1 — Qualité d'un estimateur

Pour $x$ fixé, $\hat f_n(x)$ est une variable aléatoire (elle dépend du tirage $P_n$). On mesure :
- le **biais** $\mathbb{E}[\hat f_n(x)] - f(x)$, la **variance** $\mathrm{Var}[\hat f_n(x)]$,
- l'**erreur quadratique moyenne** $\mathrm{MSE} = \mathbb{E}[(\hat f_n(x) - f(x))^2] = \mathrm{Biais}^2 + \mathrm{Variance}$.

Dans ce TP on estime ces quantités par **Monte-Carlo** : on retire $P_n$ de nombreuses fois à partir d'une loi connue. La fonction ci-dessous est fournie.

In [ ]:
def monte_carlo_error(fit, sampler, f_true, x_eval, n, n_rep=100, seed=0):
    """fit(sample) -> fonction densité ; sampler(n, rng) -> échantillon ; renvoie biais², variance, MSE
    moyennés sur les points x_eval."""
    r = np.random.default_rng(seed)
    est = np.array([fit(sampler(n, r))(x_eval) for _ in range(n_rep)])      # (n_rep, m)
    bias2 = (est.mean(0) - f_true(x_eval)) ** 2
    var = est.var(0)
    return bias2.mean(), var.mean(), (bias2 + var).mean()

## Partie 2 — Le modèle gaussien

$\hat\mu_n = \frac1n\sum p_i$, $\hat\Sigma_n = \frac1n\sum (p_i - \hat\mu_n)(p_i - \hat\mu_n)^\top$ (MLE), puis $\hat f_n = \mathcal N(\hat\mu_n, \hat\Sigma_n)$.
- Si les données sont gaussiennes : erreur $O_P(\sqrt{d/n})$, la vitesse paramétrique.
- Sinon : $\hat f_n \to \bar f = \mathcal N(\mu, \Sigma) \neq f$ — **biais constant**, qui ne disparaît jamais.

### Exercice 2.1 — `fit_gaussian` et `gaussian_logpdf`
Calculez la log-densité de façon stable avec une **décomposition de Cholesky** $\Sigma = LL^\top$ : $\log f(x) = -\frac12\|L^{-1}(x-\mu)\|^2 - \sum_i \log L_{ii} - \frac d2\log 2\pi$.

In [ ]:
from scipy.linalg import cholesky, solve_triangular

def fit_gaussian(X):
    X = np.asarray(X, float).reshape(len(X), -1)
    # TODO : moyenne, puis Xc.T @ Xc / n (estimateur MLE, pas n − 1)
    raise NotImplementedError("À compléter")


def gaussian_logpdf(X, mu, Sigma):
    X = np.asarray(X, float).reshape(-1, len(mu))
    # TODO : L = cholesky(Sigma, lower=True) ; z = solve_triangular(L, (X − μ)ᵀ, lower=True)
    raise NotImplementedError("À compléter")

In [ ]:
# Test
X = rng.multivariate_normal([1, -2, 0], [[2, .5, 0], [.5, 1, .3], [0, .3, .5]], size=500)
mu, S = fit_gaussian(X)
check_close(S, np.cov(X.T, bias=True), "covariance MLE (1/n)")
check_close(gaussian_logpdf(X[:5], mu, S), stats.multivariate_normal(mu, S).logpdf(X[:5]), "logpdf = scipy")

### Expérience 2.2 — Vitesse $1/\sqrt n$… ou biais constant

On estime $f(0)$ pour des données $\mathcal N(0,1)$ puis pour des données Student-$t$ à 3 degrés de liberté (standardisées : variance 1, queues épaisses — un modèle classique des rendements).

In [ ]:
ns = [50, 200, 1000, 5000, 20000]
t3 = stats.t(df=3, scale=1 / np.sqrt(3))       # variance 1
cases = {"données gaussiennes": (lambda n, r: r.standard_normal(n), stats.norm.pdf),
         "données Student-t(3)": (lambda n, r: t3.rvs(n, random_state=r), t3.pdf)}
fit_g = lambda s: (lambda x: np.exp(gaussian_logpdf(x, *fit_gaussian(s))))
fig, ax = plt.subplots()
for i, (name, (sampler, f)) in enumerate(cases.items()):
    err = [np.sqrt(monte_carlo_error(fit_g, sampler, f, np.array([0.0]), n, n_rep=200)[2]) for n in ns]
    ax.loglog(ns, err, marker="o", label=name, color=PALETTE[i])
ax.loglog(ns, 0.4 / np.sqrt(ns), ls="--", color=GRAY, lw=1, label="pente −1/2")
ax.set_xlabel("n"); ax.set_ylabel("racine du MSE en x = 0"); ax.legend()
ax.set_title("Modèle gaussien : convergent si le modèle est vrai, biaisé sinon")
plt.show()

### Application 2.3 — Les queues du S&P 500

Ajustons une gaussienne aux rendements quotidiens du SPY (2000–2024) et comparons la fréquence des journées extrêmes à ce que prédit le modèle.

In [ ]:
spy, src = tp_data.load_index_returns("SPY", start="2000-01-01")
r = spy.to_numpy()
print("source :", src, "| n =", len(r))
mu_r, var_r = fit_gaussian(r)
sd_r = np.sqrt(var_r[0, 0]); mu_r = mu_r[0]
rows = []
for k in [2, 3, 4, 5]:
    emp = np.mean(r < mu_r - k * sd_r)
    gau = stats.norm.cdf(-k)
    rows.append((f"< −{k}σ", emp, gau, emp / gau, 1 / (gau * 252)))
pd.DataFrame(rows, columns=["journée", "fréquence observée", "prob. gaussienne", "rapport",
                            "période de retour gaussienne (années)"]).set_index("journée")

In [ ]:
xs = np.linspace(r.min(), r.max(), 400)
fig, ax = plt.subplots()
ax.hist(r, bins=200, density=True, color=BLUE, alpha=0.35, label="histogramme")
ax.plot(xs, stats.norm.pdf(xs, mu_r, sd_r), color=ORANGE, label="gaussienne ajustée")
ax.set_yscale("log"); ax.set_ylim(1e-2, None); ax.set_xlabel("rendement quotidien")
ax.set_title("Échelle log : la gaussienne sous-estime massivement les queues"); ax.legend()
plt.show()

## Partie 3 — Mélanges de gaussiennes (GMM) et algorithme EM

$\bar f(x) = \sum_{l=1}^r \xi_l\,\phi_{\mu_l,\Sigma_l}(x)$, $\xi_l \ge 0$, $\sum\xi_l = 1$. Les mélanges finis approchent toute densité continue à support compact (Bacharoglou 2010) : ils suppriment le biais du modèle gaussien quand $r$ est assez grand. Le MLE n'a pas de forme close et la log-vraisemblance **n'est pas concave** : on utilise l'algorithme **EM**.

- **E-step** : responsabilités $\gamma_{il} = \dfrac{\xi_l\,\phi_l(p_i)}{\sum_m \xi_m\,\phi_m(p_i)}$ (calculées en log avec `logsumexp`).
- **M-step** : $N_l = \sum_i\gamma_{il}$, $\ \xi_l = N_l/n$, $\ \mu_l = \frac1{N_l}\sum_i\gamma_{il}p_i$, $\ \Sigma_l = \frac1{N_l}\sum_i\gamma_{il}(p_i-\mu_l)(p_i-\mu_l)^\top$ (+ petite régularisation diagonale).

EM fait **croître la log-vraisemblance à chaque itération**, mais converge vers un **maximum local**. (k-means/Lloyd est le cas limite d'EM avec covariances $\varepsilon I$, $\varepsilon\to0$, et affectations « dures ».)

### Exercice 3.1 — `gmm_em`

In [ ]:
def gmm_em(X, r, n_iter=500, tol=1e-9, seed=0, reg=1e-6):
    """Retourne (weights (r,), means (r, d), covs (r, d, d), historique de la log-vraisemblance moyenne)."""
    X = np.asarray(X, float).reshape(len(X), -1)
    n, d = X.shape
    rr = np.random.default_rng(seed)
    eps = reg * np.trace(np.atleast_2d(np.cov(X.T))) / d       # régularisation relative à l'échelle
    weights = np.full(r, 1 / r)
    means = X[rr.choice(n, r, replace=False)].copy()
    covs = np.array([np.atleast_2d(np.cov(X.T)) + eps * np.eye(d) for _ in range(r)])
    hist = []
    for it in range(n_iter):
        # TODO : E-step (log-responsabilités, logsumexp), historique, M-step, critère d'arrêt
        raise NotImplementedError("À compléter")
    return weights, means, covs, hist


def gmm_logpdf(X, weights, means, covs):
    X = np.asarray(X, float).reshape(-1, means.shape[1])
    return logsumexp(np.stack([np.log(w) + gaussian_logpdf(X, m, c)
                               for w, m, c in zip(weights, means, covs)], 1), axis=1)

In [ ]:
# Tests
true_means = np.array([[0, 0], [5, 5], [0, 6]])
Xg = np.vstack([rng.multivariate_normal(m, np.eye(2) * s, n) for m, s, n in
                zip(true_means, [1.0, 0.5, 1.5], [300, 200, 500])])
fits = [gmm_em(Xg, 3, seed=s) for s in range(5)]
w, m, c, h = max(fits, key=lambda f: f[3][-1])
check(np.all(np.diff(h) >= -1e-10), "EM : la log-vraisemblance ne décroît jamais")
m_sorted = m[np.lexsort(m.T[::-1])]
check_close(m_sorted, true_means[np.lexsort(true_means.T[::-1])], "moyennes retrouvées", atol=0.3)
sk = GaussianMixture(3, n_init=5, random_state=0).fit(Xg)
check(abs(h[-1] - sk.score(Xg)) < 1e-3, f"log-vraisemblance = sklearn ({h[-1]:.4f} vs {sk.score(Xg):.4f})")

### Expérience 3.2 — Maxima locaux et choix de $r$ par BIC

Plusieurs initialisations mènent à des maxima locaux différents (d'où `n_init` dans scikit-learn). Pour choisir $r$, la log-vraisemblance augmente toujours avec $r$ ; on la pénalise par le nombre de paramètres $p$ : $\mathrm{BIC} = -2\,n\,\overline{\ell} + p\log n$ (à minimiser), avec $p = (r-1) + rd + r\,d(d+1)/2$.

In [ ]:
def bic(X, r, n_init=5):
    X = np.asarray(X, float).reshape(len(X), -1); n, d = X.shape
    # TODO : meilleur de n_init appels à gmm_em ; p = (r − 1) + r d + r d(d+1)/2 ; hist[-1] = log-vraisemblance moyenne
    raise NotImplementedError("À compléter")

finals = [gmm_em(Xg, 3, seed=s)[3][-1] for s in range(20)]
print("log-vraisemblances finales (20 initialisations) :", np.round(sorted(set(np.round(finals, 4))), 4))
bics = {r_: bic(Xg, r_)[0] for r_ in range(1, 8)}
check(min(bics, key=bics.get) == 3, "le BIC retrouve r = 3")
pd.Series(bics, name="BIC").to_frame().T

### Application 3.3 — Deux régimes dans les rendements du S&P 500

Un GMM 1D à deux composantes sur les rendements quotidiens donne typiquement une composante **calme** (poids élevé, faible variance) et une composante **turbulente** (poids faible, forte variance). La responsabilité $\gamma_{i,\text{turb}}$ est la probabilité a posteriori qu'un jour donné « vienne » du régime turbulent ; lissée sur un mois, elle dessine les crises.

In [ ]:
w2, m2, c2, h2 = max((gmm_em(r, 2, seed=s) for s in range(5)), key=lambda f: f[3][-1])
sd2 = np.sqrt(c2[:, 0, 0]); turb = int(np.argmax(sd2))
print(pd.DataFrame({"poids": w2, "moyenne (annualisée)": m2[:, 0] * 252,
                    "vol (annualisée)": sd2 * np.sqrt(252)}, index=["comp. 0", "comp. 1"]))
logp = np.stack([np.log(w2[l]) + gaussian_logpdf(r, m2[l], c2[l]) for l in range(2)], 1)
post_turb = pd.Series(np.exp(logp[:, turb] - logsumexp(logp, axis=1)), index=spy.index)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(spy.index, np.cumsum(np.log1p(r)), color=INK2, lw=1); axes[0].set_title("SPY — log-prix cumulé")
axes[1].plot(post_turb.rolling(21).mean(), color=RED, lw=1.2)
axes[1].set_title("P(régime turbulent | rendement), moyenne mobile 21 jours"); axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## Partie 4 — Histogrammes

On découpe le domaine en une grille de $N^d$ cellules et $\hat f_n(x) = \dfrac{\#\{p_i\in\mathrm{Cell}(x)\}}{n\,\mathrm{Vol}(\mathrm{Cell}(x))}$.
Compromis **biais-variance** contrôlé par $N$ : biais $O(\sqrt d/N)$ (cellules larges), variance $O(N^d/n)$ (peu de points par cellule). Optimum $N \propto n^{1/(d+2)}$, $\mathrm{MSE} = O(n^{-2/(d+2)})$.

### Exercice 4.1 — `fit_histogram`
Domaine : la boîte $[lo, hi)$ (vecteurs de taille $d$), grille $N$ par axe. Les points hors de la boîte ont une densité nulle. Utilisez `np.ravel_multi_index` et `np.bincount`.

In [ ]:
def fit_histogram(X, N, lo, hi):
    X = np.asarray(X, float).reshape(len(X), -1)
    n, d = X.shape
    lo, hi = np.broadcast_to(np.asarray(lo, float), (d,)), np.broadcast_to(np.asarray(hi, float), (d,))
    h = (hi - lo) / N
    # TODO : indices de cellule, comptage, puis fonction d'évaluation
    raise NotImplementedError("À compléter")

In [ ]:
# Test
X = rng.random((10000, 2))
f = fit_histogram(X, 10, [0, 0], [1, 1])
check(abs(f(rng.random((2000, 2))).mean() - 1) < 0.05, "densité uniforme ≈ 1 sur le carré")
x1 = rng.standard_normal(5000)
f1 = fit_histogram(x1, 40, -5, 5)
grid = np.linspace(-5, 5, 4001)[:-1] + 1.25e-3
check(abs(f1(grid).sum() * 2.5e-3 - np.mean(np.abs(x1) < 5)) < 1e-3, "l'histogramme intègre à la masse dans la boîte")

### Expérience 4.2 — Biais, variance et choix de $N$ ($d = 1$, $n = 1000$, loi $\mathcal N(0,1)$)

On mesure biais², variance et MSE moyennés sur une grille de points, en fonction de $N$ (nombre de cellules sur $[-4, 4]$).

In [ ]:
xg = np.linspace(-3, 3, 121)
Ns = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
res = [monte_carlo_error(lambda s, N=N: fit_histogram(s, N, -4, 4), lambda n, r: r.standard_normal(n),
                         stats.norm.pdf, xg, 1000, n_rep=200) for N in Ns]
res = np.array(res)
fig, ax = plt.subplots()
for j, lbl in enumerate(["biais²", "variance", "MSE"]):
    ax.loglog(Ns, res[:, j], marker="o", label=lbl, color=PALETTE[j])
ax.set_xlabel("N (nombre de cellules)"); ax.set_title("Histogramme : compromis biais-variance (n = 1000)")
ax.legend(); plt.show()
print("N optimal ≈", Ns[int(np.argmin(res[:, 2]))])

## Partie 5 — Estimateurs à noyau (KDE)

On place une copie d'une densité élémentaire (le **noyau**) sur chaque observation. Pour un noyau isotrope radial de **profil** $k$ et de **fenêtre** $\sigma$ :
$$\hat f_n(x) = \frac{c_{k,d}}{n\,\sigma^d}\sum_{i=1}^n k\!\left(\frac{\|x - p_i\|^2}{\sigma^2}\right),\qquad c_{k,d} = \Big(\int_{\mathbb R^d}k(\|u\|^2)\,du\Big)^{-1}.$$

| noyau | profil $k(t)$ | $c_{k,d}$ |
|---|---|---|
| uniforme | $\mathbb 1_{t\le1}$ | $1/\mathrm{Vol}(B_d)$ |
| Epanechnikov | $(1-t)\,\mathbb 1_{t\le1}$ | $\frac{d+2}{2\,\mathrm{Vol}(B_d)}$ |
| gaussien | $e^{-t/2}$ | $(2\pi)^{-d/2}$ |

avec $\mathrm{Vol}(B_d) = \pi^{d/2}/\Gamma(d/2+1)$. Biais $O(\sigma^2)$, variance $O(1/(n\sigma^d))$ → $\sigma_{\mathrm{opt}}\propto n^{-1/(d+4)}$ et $\mathrm{MSE} = O(n^{-4/(d+4)})$.

### Exercice 5.1 — `kde`

In [ ]:
def unit_ball_volume(d):
    return np.pi ** (d / 2) / gamma(d / 2 + 1)


def kde(P, X, sigma, kernel="gaussian"):
    """Évalue la KDE construite sur P (n x d) aux points X (m x d)."""
    P = np.asarray(P, float).reshape(len(P), -1); n, d = P.shape
    X = np.asarray(X, float).reshape(-1, d)
    # TODO : t = ||x - p||² / σ² (matrice m x n), profil, constante, moyenne
    raise NotImplementedError("À compléter")

In [ ]:
# Tests vs scikit-learn (même normalisation, bandwidth = σ)
P = rng.normal(size=(300, 3)); Xq = rng.normal(size=(50, 3))
for ours, theirs in [("gaussian", "gaussian"), ("uniform", "tophat"), ("epanechnikov", "epanechnikov")]:
    ref = np.exp(KernelDensity(kernel=theirs, bandwidth=0.8).fit(P).score_samples(Xq))
    check_close(kde(P, Xq, 0.8, ours), ref, f"kde {ours} = sklearn", rtol=1e-6, atol=1e-12)
grid = np.linspace(-6, 6, 3001)
check(abs(kde(rng.normal(size=200), grid, 0.5, "epanechnikov").sum() * (grid[1] - grid[0]) - 1) < 1e-3,
      "la KDE 1D intègre à 1")

### Expérience 5.2 — Noyaux et fenêtre

Données bimodales inspirées du jeu *Old Faithful* (temps d'attente entre deux éruptions d'un geyser, en minutes).

In [ ]:
wait = np.concatenate([rng.normal(54, 6, 97), rng.normal(80, 6, 175)])
xs = np.linspace(30, 105, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, kern in enumerate(["uniform", "epanechnikov", "gaussian"]):
    axes[0].plot(xs, kde(wait, xs, 4.0, kern), label=kern, color=PALETTE[i])
axes[0].set_title("Trois noyaux, σ = 4"); axes[0].legend()
for i, s in enumerate([1, 3, 10]):
    axes[1].plot(xs, kde(wait, xs, s), label=f"σ = {s}", color=PALETTE[i])
axes[1].set_title("Noyau gaussien : sous-lissage vs sur-lissage"); axes[1].legend()
for ax in axes:
    ax.plot(wait, np.full_like(wait, -0.002), "|", color=INK2, alpha=0.5); ax.set_xlabel("minutes")
plt.tight_layout(); plt.show()

**À retenir :** le choix du noyau importe peu ; **le choix de σ est crucial**. Grand σ : biais fort (on efface la bimodalité), variance faible. Petit σ : biais faible, variance forte (pics parasites).

### Exercice 5.3 — Choisir σ : règle de Silverman et validation croisée

- **Silverman** (1D, noyau gaussien) : $\sigma = 1.06\,\hat s\, n^{-1/5}$ — optimale si les données sont gaussiennes, trop large pour une densité multimodale.
- **Vraisemblance leave-one-out** : $\mathrm{CV}(\sigma) = \frac1n\sum_i \log \hat f_{-i}(p_i)$ où $\hat f_{-i}$ est la KDE construite sans $p_i$ ; on maximise sur une grille de σ. Astuce : avec la matrice $K_{ij} = K_\sigma(p_i - p_j)$, $\hat f_{-i}(p_i) = \frac{1}{n-1}\sum_{j\ne i}K_{ij}$.

In [ ]:
def silverman(x):
    x = np.asarray(x, float)
    return 1.06 * x.std() * len(x) ** (-1 / 5)


def loo_loglik(x, sigma):
    """Log-vraisemblance leave-one-out moyenne d'une KDE gaussienne 1D."""
    x = np.asarray(x, float)
    # TODO : matrice K des noyaux gaussiens, diagonale mise à 0, moyenne de log(K.sum(1) / (n − 1))
    raise NotImplementedError("À compléter")

sigmas = np.geomspace(0.5, 20, 40)
cv = [loo_loglik(wait, s) for s in sigmas]
s_cv = sigmas[int(np.argmax(cv))]
check(s_cv < silverman(wait), "sur des données bimodales, la CV choisit une fenêtre plus fine que Silverman")
print(f"σ Silverman = {silverman(wait):.2f}  |  σ CV = {s_cv:.2f}")
fig, ax = plt.subplots()
ax.semilogx(sigmas, cv, marker="o", ms=3); ax.axvline(s_cv, color=GRAY, ls="--", lw=1)
ax.set_xlabel("σ"); ax.set_ylabel("log-vraisemblance LOO"); ax.set_title("Validation croisée de la fenêtre")
plt.show()

### Expérience 5.4 — Vitesses de convergence et malédiction de la dimension

Données $\mathcal N(0, I_d)$, estimation de $f(0)$ ; pour chaque $n$ on prend le σ « oracle » qui minimise le MSE (on mesure ainsi la **meilleure vitesse atteignable**). Théorie : pente $-4/(d+4)$ sur le MSE, soit $-0.8$, $-0.67$, $-0.44$ pour $d = 1, 2, 5$ (vs $-1$ pour un modèle paramétrique correct).

In [ ]:
ns = [100, 300, 1000, 3000, 10000]
slopes = {}
fig, ax = plt.subplots()
for i, d in enumerate([1, 2, 5]):
    f0 = (2 * np.pi) ** (-d / 2)
    mse_opt = []
    for n in ns:
        sig_grid = np.geomspace(0.05, 2.0, 25)
        r_ = np.random.default_rng(d * 100 + n)
        samples = [r_.standard_normal((n, d)) for _ in range(60)]
        mses = [np.mean([(kde(S, np.zeros((1, d)), s)[0] - f0) ** 2 for S in samples]) for s in sig_grid]
        mse_opt.append(min(mses))
    slopes[d] = np.polyfit(np.log(ns), np.log(mse_opt), 1)[0]
    ax.loglog(ns, mse_opt, marker="o", color=PALETTE[i], label=f"d = {d} (pente {slopes[d]:.2f}, théorie {-4/(d+4):.2f})")
ax.set_xlabel("n"); ax.set_ylabel("MSE optimal en 0"); ax.legend()
ax.set_title("KDE : plus la dimension est grande, plus la convergence est lente")
plt.show()
check(slopes[1] < slopes[2] < slopes[5], "la vitesse se dégrade avec la dimension")

## Partie 6 — Application : Value-at-Risk à 99 % du S&P 500

La **VaR à 99 %** d'une position est la perte journalière qui n'est dépassée qu'avec probabilité 1 % : c'est le quantile à 1 % de la distribution des rendements. Pour l'estimer, il faut une **densité** (ou une fonction de répartition) des rendements. On compare, en **fenêtre glissante** de 1000 jours ré-estimée tous les 20 jours :
- **gaussienne** : $\hat\mu + \hat\sigma\,\Phi^{-1}(0.01)$ ;
- **GMM** à 2 composantes : on résout $\sum_l \xi_l\,\Phi\big((q-\mu_l)/s_l\big) = 0.01$ ;
- **KDE** gaussienne (fenêtre de Silverman) : on résout $\frac1n\sum_i\Phi\big((q - p_i)/\sigma\big) = 0.01$ ;
- **historique** : quantile empirique (le « non paramétrique » le plus simple).

Le backtest compte la fréquence des **dépassements** (rendement < −VaR) : elle doit être proche de 1 %.

### Exercice 6.1

In [ ]:
def var_gaussian(x, a=0.01):
    return x.mean() + x.std() * stats.norm.ppf(a)

def var_gmm(x, a=0.01):
    # TODO : ajuster gmm_em(x, 2) ; CDF du mélange Σ ξ_l Φ((q − μ_l)/s_l) ; brentq sur [min − 10·sd, max]
    raise NotImplementedError("À compléter")

def var_kde(x, a=0.01):
    # TODO : σ = silverman(x) ; CDF = moyenne des Φ((q − x_i)/σ) ; brentq
    raise NotImplementedError("À compléter")

def var_hist(x, a=0.01):
    return np.quantile(x, a)

methods = {"gaussienne": var_gaussian, "GMM (r = 2)": var_gmm, "KDE": var_kde, "historique": var_hist}
window, step = 1000, 20
var_series = {m: pd.Series(np.nan, index=spy.index) for m in methods}
with timer("backtest VaR"):
    for t in range(window, len(r), step):
        x = r[t - window:t]
        for m, f in methods.items():
            var_series[m].iloc[t:t + step] = f(x)

realized = spy.iloc[window:]
rows = []
for m in methods:
    q = var_series[m].iloc[window:]
    exc = realized < q
    rows.append((m, exc.mean(), -q.mean(), exc.groupby(exc.index.year).mean().max()))
bt = pd.DataFrame(rows, columns=["méthode", "taux de dépassement (cible 1 %)", "VaR moyenne",
                                 "pire année (taux)"]).set_index("méthode")
observe(bt.loc["gaussienne", "taux de dépassement (cible 1 %)"] > bt.loc["historique", "taux de dépassement (cible 1 %)"],
      "la VaR gaussienne est dépassée plus souvent que la VaR historique")
bt

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(realized.index, realized, color=GRAY, lw=0.5, label="rendement")
for i, m in enumerate(["gaussienne", "historique"]):
    ax.plot(var_series[m].index[window:], var_series[m].iloc[window:], color=[ORANGE, BLUE][i], lw=1.5, label=f"VaR 99 % {m}")
exc = realized < var_series["gaussienne"].iloc[window:]
ax.scatter(realized.index[exc], realized[exc], s=10, color=RED, zorder=3, label="dépassements (gaussienne)")
ax.set_title("Backtest de VaR : les dépassements arrivent en grappes")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.show()

**Questions.**
1. Classez les méthodes selon le taux de dépassement. Pourquoi la gaussienne est-elle mauvaise ici, alors que GMM et KDE font mieux ?
2. Toutes les méthodes connaissent de mauvaises années. Qu'est-ce qu'aucune de ces densités ne modélise ?
3. Pourquoi la KDE est-elle peu adaptée pour une VaR à 99,9 % ?

> ✍️ *Votre réponse :* …

## Partie 7 — Synthèse : quel estimateur, quand ?

| Méthode | Type | Vitesse (erreur) | Paramètre | Limite principale | Quand l'utiliser |
|---|---|---|---|---|---|
| **Gaussienne** | paramétrique | $O(\sqrt{d/n})$ | — | **biais** si la loi n'est pas gaussienne | peu de données, grande dimension, première approximation ; covariance comme objet central (portefeuille) |
| **GMM / EM** | paramétrique | $O(\sqrt{d/n})$ (vers le meilleur mélange) | nombre $r$ (BIC) | maxima locaux, coût, singularités | densités multimodales/asymétriques, **régimes**, clustering « soft » (généralise k-means) |
| **Histogramme** | non paramétrique | $O(n^{-1/(d+2)})$ | nombre $N$ de cellules | discontinu, malédiction de la dimension | exploration 1D–2D rapide, très grands $n$ |
| **KDE** | non paramétrique | $O(n^{-2/(d+4)})$ | fenêtre σ (CV, Silverman) | malédiction de la dimension, coût $O(n)$ par évaluation | densité lisse en petite dimension (≤ 3–5), visualisation, mode-seeking (mean-shift) |

**Règle pratique :** paramétrique si l'on croit (au moins approximativement) au modèle ou si $d$ est grand ; non paramétrique en petite dimension avec beaucoup de données ; le GMM est souvent le bon compromis.